<a href="https://colab.research.google.com/github/nadynstary/Astria-s-Song/blob/main/Sonification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
image_to_sound.py  -  turn an image into a .wav file

Install:  pip install numpy pillow scipy

Usage:
  python image_to_sound.py my_image.jpg                 # scan mode (default)
  python image_to_sound.py my_image.jpg --mode color    # melody from colors
  python image_to_sound.py my_image.jpg --duration 12 --out result.wav
  python image_to_sound.py my_image.jpg --invert        # dark pixels = loud (scan mode)
"""
import argparse
import colorsys
import numpy as np
from PIL import Image
from scipy.io import wavfile

SR = 22050  # sample rate


def normalize(x):
    peak = np.max(np.abs(x))
    return x / peak * 0.9 if peak > 0 else x


# ---------------------------------------------------------------- scan mode
def sonify_scan(img, duration, fmin=100, fmax=5000, n_rows=96, invert=False):
    """x = time, y = pitch (top is high), brightness = loudness."""
    pix_w = max(2, int(duration * 40))
    g = img.convert("L").resize((pix_w, n_rows), Image.LANCZOS)
    a = np.asarray(g, dtype=float) / 255.0
    if invert:
        a = 1.0 - a
    a = a[::-1]  # row 0 = bottom = lowest frequency

    n = int(duration * SR)
    t = np.arange(n) / SR
    xs = np.linspace(0, pix_w - 1, n)
    grid = np.arange(pix_w)
    freqs = np.geomspace(fmin, fmax, n_rows)  # log spacing = how we hear pitch
    rng = np.random.default_rng(0)

    out = np.zeros(n)
    for i, f in enumerate(freqs):
        env = np.interp(xs, grid, a[i])  # smooth loudness over time
        if env.max() < 0.02:
            continue
        out += env * np.sin(2 * np.pi * f * t + rng.uniform(0, 2 * np.pi))
    return normalize(out)


# --------------------------------------------------------------- color mode
PENTATONIC = [0, 2, 4, 7, 9]  # major pentatonic semitone steps


def hue_to_freq(h, base=220.0, octaves=3):
    steps = len(PENTATONIC) * octaves
    idx = min(int(h * steps), steps - 1)
    octave, degree = divmod(idx, len(PENTATONIC))
    semitones = 12 * octave + PENTATONIC[degree]
    return base * 2 ** (semitones / 12)


def sonify_color(img, duration, n_notes=48):
    """Each vertical slice -> one note. hue=pitch, value=volume, saturation=harmonics."""
    # BOX resize to (n_notes x 1) = average color of each slice
    small = img.convert("RGB").resize((n_notes, 1), Image.BOX)
    pixels = np.asarray(small)[0] / 255.0

    note_len = duration / n_notes
    n_per = int(note_len * SR)
    t = np.arange(n_per) / SR
    # attack/decay envelope so notes don't click
    env = np.minimum(1, np.minimum(t / 0.01, (note_len - t) / (note_len * 0.4 + 1e-9)))
    env = np.clip(env, 0, 1)

    notes = []
    for r, g, b in pixels:
        h, s, v = colorsys.rgb_to_hsv(r, g, b)
        f = hue_to_freq(h)
        n_harm = 1 + int(s * 5)  # more saturated = brighter/richer tone
        tone = sum(np.sin(2 * np.pi * f * (k + 1) * t) / (k + 1) for k in range(n_harm))
        notes.append(tone * env * (0.15 + 0.85 * v))
    return normalize(np.concatenate(notes))


# --------------------------------------------------------------------- main
def main():
    p = argparse.ArgumentParser()
    p.add_argument("image")
    p.add_argument("--mode", choices=["scan", "color"], default="scan")
    p.add_argument("--duration", type=float, default=8.0, help="seconds")
    p.add_argument("--out", default="output.wav")
    p.add_argument("--invert", action="store_true", help="scan mode: dark = loud")
    args = p.parse_args()

    img = Image.open(args.image)
    if args.mode == "scan":
        audio = sonify_scan(img, args.duration, invert=args.invert)
    else:
        audio = sonify_color(img, args.duration)

    wavfile.write(args.out, SR, (audio * 32767).astype(np.int16))
    print(f"Saved {args.out} ({args.mode} mode, {args.duration}s)")

if __name__ == "__main__":
    pass


In [ ]:
import numpy as np
from PIL import Image
from scipy.io import wavfile
from IPython.display import Audio, display

SR = 22050
SCALE = [0, 2, 4, 7, 9, 11]  # C D E G A B

def bell_note(freq, dur, vel=1.0):
    t = np.arange(int(dur * SR)) / SR
    tone = (np.sin(2*np.pi*freq*t)
            + 0.25*np.sin(2*np.pi*2*freq*t) * np.exp(-t*3)
            + 0.08*np.sin(2*np.pi*3*freq*t) * np.exp(-t*5))
    env = np.minimum(1, t/0.004) * np.exp(-t*1.6)
    return tone * env * vel

def sonify_radial(img, duration=30, n_steps=120, notes_per_step=3, threshold=0.18,
                  low_note=55, high_note=91, center=(0.5, 0.5), outside_in=True,
                  invert=False, tail=3.0, n_angles=360):
    g = img.convert("L")
    g.thumbnail((500, 500))
    a = np.asarray(g, dtype=float) / 255.0
    if invert:
        a = 1.0 - a
    H, W = a.shape
    cx, cy = center[0] * (W - 1), center[1] * (H - 1)
    r_max = max(np.hypot(cx, cy), np.hypot(W - 1 - cx, cy),
                np.hypot(cx, H - 1 - cy), np.hypot(W - 1 - cx, H - 1 - cy))
    r_max *= 0.75

    midis = [m for m in range(low_note, high_note + 1) if m % 12 in SCALE]
    freqs = [440 * 2 ** ((m - 69) / 12) for m in midis]
    angles = np.linspace(0, 2*np.pi, n_angles, endpoint=False)

    step_t = duration / n_steps
    out = np.zeros((int((duration + tail) * SR), 2))

    for j in range(n_steps):
        prog = j / n_steps
        r = r_max * (1 - prog) if outside_in else r_max * prog
        r = max(r, 2)
        xs = np.round(cx + r*np.cos(angles)).astype(int)
        ys = np.round(cy + r*np.sin(angles)).astype(int)
        ok = (xs >= 0) & (xs < W) & (ys >= 0) & (ys < H)
        b = np.zeros(n_angles)
        b[ok] = a[ys[ok], xs[ok]]
        b = np.convolve(np.r_[b[-1], b, b[0]], np.ones(3)/3, mode="valid")

        peaks = (b > np.roll(b, 1)) & (b >= np.roll(b, -1)) & (b > threshold)
        idxs = np.where(peaks)[0]
        idxs = idxs[np.argsort(b[idxs])[-notes_per_step:]]

        frac = r / r_max                              # 1 = edge, 0 = center
        n_idx = int(round(frac * (len(freqs) - 1)))   # edge = high pitch
        closeness = 1 - frac if outside_in else frac
        gain = 0.12 + 0.88 * closeness ** 1.5         # louder near the center

        for k in idxs:
            pan = np.cos(angles[k])                   # left <-> right by angle
            th = (pan + 1) * np.pi / 4
            note = bell_note(freqs[n_idx], 2.5, vel=b[k] * gain)
            s = int(j * step_t * SR)
            e = min(s + len(note), len(out))
            out[s:e, 0] += note[:e - s] * np.cos(th)
            out[s:e, 1] += note[:e - s] * np.sin(th)

    for d, gdb in ((int(0.23*SR), 0.30), (int(0.37*SR), 0.20)):  # echo
        out[d:] += gdb * out[:-d].copy()
    return out / (np.max(np.abs(out)) + 1e-9) * 0.9

In [ ]:
from google.colab import files
from IPython.display import Audio, display
import numpy as np
from PIL import Image
from scipy.io import wavfile

uploaded = files.upload()
print("uploaded:", list(uploaded.keys()))

path = list(uploaded.keys())[0]
img = Image.open(path)
print("image size:", img.size)

audio = sonify_scan(img, duration=10)
print("audio samples:", len(audio), "| max:", float(np.max(np.abs(audio))))

wavfile.write("scan.wav", SR, (audio * 32767).astype(np.int16))
display(Audio("scan.wav"))

Saving 5.jpg to 5 (1).jpg
uploaded: ['5 (1).jpg']
image size: (678, 452)
audio samples: 220500 | max: 0.9


In [ ]:
audio = sonify_radial(img)   # takes ~10-20 seconds to compute
wavfile.write("radial.wav", SR, (audio * 32767).astype(np.int16))
display(Audio("radial.wav"))